# 10 · 🔥 Chaos Lab: Matando um NodeManager no Meio do Job (Caso C)

**Teoria**: docs/02-rdds-linhagem-particoes.md ("Linhagem: Tolerância a Falhas Sem Replicação")

**Pré-requisito**: `make up-hadoop` ainda em execução, com **ambos**
`nodemanager1` e `nodemanager2` saudáveis (`make status`).

---

🎯 **Objetivo**: vamos executar um Job deliberadamente lento em uma thread de background,
matar um NodeManager durante sua execução, e observar o YARN **reagendar as Tasks
perdidas** no NodeManager sobrevivente — recomputando apenas o que foi perdido, via
**linhagem**, exatamente como docs/02 descreve.

📌 **Este é o teste de fogo da tolerância a falhas do Spark.** Diferente de sistemas
tradicionais que precisam de replicação para se recuperar de falhas, o Spark usa a
**linhagem** (lineage) dos RDDs/DataFrames para **recalcular** partitions perdidas.

🔁 **Inspiração**: este experimento espelha a demonstração `restart-datanode3`
do `cdn-hadoop-lab`, mas para o mecanismo de **processamento** (YARN/Spark) em vez
de armazenamento (HDFS).

⚠️ **Aviso**: vamos usar uma UDF Python (sabidamente lenta) de propósito — é um dos
poucos momentos do laboratório onde a ineficiência intencional é a ferramenta certa
para criar uma janela de tempo grande o suficiente para intervir.

In [ ]:
import subprocess
import sys
import threading
import time

# Importa funções auxiliares: get_yarn_session cria SparkSession conectada ao YARN
# layer_path constrói caminhos HDFS para os dados
sys.path.insert(0, "../scripts")
from lab_utils import get_yarn_session, layer_path
from pyspark.sql.functions import col

# Cria SparkSession conectada ao cluster YARN (client mode, como no Lab 08)
spark = get_yarn_session("10-chaos-lab")

# Carrega os dados de vendas do HDFS (via webhdfs://) que enviamos no Lab 08
vendas = spark.read.parquet(layer_path("hdfs", "bronze", "vendas"))

# .count() força a leitura para confirmar que os dados estão acessíveis
print(f"Working set: {vendas.count():,} rows across HDFS")

### 📌 Dados carregados — o que temos?

Temos o dataset `vendas` carregado do HDFS. Este DataFrame será a base do nosso
experimento de tolerância a falhas.

🧠 **Por que carregar do HDFS é importante aqui?**
- Os dados estão armazenados em **blocos replicados** nos DataNodes (fator 2)
- Quando um NodeManager morrer, as Tasks que estavam rodando nele serão perdidas
- Mas os dados **continuam disponíveis** no HDFS (no outro DataNode)
- O Spark simplesmente **relerá os blocos** do DataNode sobrevivente e **recalculará**
  as partitions perdidas via linhagem

> ⚠️ **Importante**: a tolerância a falhas do Spark **não depende de replicação de
> processamento** — depende da **linhagem**. Se um executor morre, outro assume e
> re-executa as mesmas operações a partir dos dados de entrada.

## Um Job deliberadamente lento

Para que tenhamos tempo de intervir (matar o NodeManager), precisamos de um Job
que demore **vários segundos** para completar.

🔧 **A estratégia:**
1. Criamos uma UDF (User Defined Function) em Python puro que dorme 10ms por linha
2. UDFs Python são notoriamente lentas (docs/05 explica por quê — serialização
   Python-JVM a cada linha)
3. Aplicamos esta UDF a 20.000 linhas → ~200 segundos de execução
4. Executamos o Job em uma **thread separada** para não travar o notebook

> 💡 **Por que UDF Python e não uma operação nativa?** Operações nativas do Spark
> (`.filter()`, `.groupBy()`) executam dentro da JVM e são extremamente rápidas —
> não nos dariam a janela de tempo necessária para o experimento.

In [ ]:
import time as _time

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType


# UDF (User Defined Function) que introduz um delay artificial
# @udf(returnType=...) registra esta função Python como uma UDF Spark
@udf(returnType=DoubleType())
def slow_identity(valor):
    # Dorme 10ms por linha para tornar o Job deliberadamente lento
    # Em produção isso seria um anti-pattern — aqui é a ferramenta certa
    _time.sleep(0.01)  # 10ms/row -> deliberately slow, for demo purposes only
    return valor


# Dicionário para compartilhar o resultado entre a thread e o notebook principal
result_holder = {}


def run_slow_job():
    # Executa uma agregação lenta em background
    start = _time.perf_counter()
    total = (
        vendas
        .limit(20_000)  # Limita a 20k linhas para o experimento
        .withColumn("valor_slow", slow_identity(col("valor")))  # Aplica UDF lenta
        .agg({"valor_slow": "sum"})  # Soma os valores
        .collect()  # Action: dispara o Job
    )
    result_holder["seconds"] = _time.perf_counter() - start
    result_holder["total"] = total[0][0]


# Inicia o Job em uma thread separada para não travar o notebook
# Enquanto o Job roda, podemos executar outras células
job_thread = threading.Thread(target=run_slow_job)
job_thread.start()

# Mensagem para o usuário acompanhar na UI do YARN
print("Job launched in the background — check http://localhost:8088 for the Application.")
print("Wait a few seconds for both NodeManagers to receive tasks, then run the next cell.")

### 🧠 Por que usar uma thread separada?

O Jupyter Notebook executa células sequencialmente. Se o Job lento estivesse na
célula principal, ela bloquearia até terminar — e não conseguiríamos executar o
comando `docker kill` no meio da execução.

**Solução**: rodar o Job em uma `Thread` separada. Agora:
- O Job Spark roda em background (em paralelo)
- Podemos executar a célula de "matar o NodeManager" enquanto o Job está ativo
- A UI do YARN em http://localhost:8088 mostra a Application em RUNNING

📌 **Timing**: dê **~8 segundos** para o Job realmente começar a executar
(negociação YARN + inicialização dos executores + início do processamento)
antes de prosseguir.

## Mate um NodeManager enquanto o Job está executando

⏳ **Antes de executar esta célula:**
1. Verifique http://localhost:8088 — a Application deve estar em RUNNING
2. Veja se há containers ativos em ambos os NodeManagers
3. Só então execute a célula abaixo

⚠️ **O momento do kill é crítico**: se o Job ainda não tiver começado a executar,
o YARN simplesmente reagenda tudo no NodeManager restante e o experimento
perde a graça. Se já tiver terminado, não há o que matar.

In [ ]:
# Aguarda alguns segundos para garantir que o Job está realmente executando
time.sleep(8)

# Mata o container nodemanager1 de forma forçada (SIGKILL)
# Isso simula uma falha abrupta de hardware — sem graceful shutdown
print("💥 Killing nodemanager1 mid-job...")
subprocess.run(["docker", "kill", "nodemanager1"], check=True)

print("nodemanager1 is down. Watch the ResourceManager UI: the Application")
print("should keep running, rescheduling nodemanager1's lost tasks onto")
print("nodemanager2 — no data was lost, because Spark recomputes lost")
print("partitions from lineage (docs/02), it doesn't need a backup copy.")

### ⏳ O que observar no YARN agora

Assim que o `nodemanager1` morre:

1. 🔴 **ResourceManager detecta a falha**: após ~10s (timeout de heartbeat), o RM
   marca o NodeManager como `LOST`
2. 🔄 **Tasks perdidas são identificadas**: todas as Tasks que estavam executando
   no nodemanager1 são marcadas como `FAILED`
3. ✅ **Reagendamento automático**: o ApplicationMaster solicita ao ResourceManager
   que reexecute essas Tasks no `nodemanager2` (o único sobrevivente)
4. ♻️ **Recomputação via linhagem**: o Spark não tinha uma cópia de backup dos
   resultados parciais — ele simplesmente **recalcula** as partitions perdidas
   a partir dos dados de entrada no HDFS

🧠 **Isso é o poder da linhagem:**
- Sem replicação de estado intermediário
- Sem checkpoint manual
- Sem custo de armazenamento para tolerância a falhas
- Apenas o DAG (Directed Acyclic Graph) de operações é mantido

> 💡 Na UI do ResourceManager, você verá o número de containers diminuir e depois
> se estabilizar — o YARN está se adaptando à nova capacidade do cluster.

In [ ]:
# Aguarda a thread do Job terminar (bloqueante)
# O Job pode estar rodando apenas no nodemanager2 agora
job_thread.join()

# Exibe o resultado — o Job deve ter completado com sucesso
# APESAR de ter perdido um NodeManager no meio do caminho
print(f"Job finished in {result_holder['seconds']:.1f}s despite losing a NodeManager mid-flight.")
print(f"Result: {result_holder['total']:.2f}")

### 🎯 Lições de tolerância a falhas

✅ **O Job completou!** Mesmo com a perda de um NodeManager inteiro no meio da
execução, o Spark conseguiu:
- Detectar a falha (via timeout do heartbeat)
- Reagendar as Tasks perdidas no nó restante
- Recomputar as partitions perdidas via linhagem
- Produzir o **mesmo resultado** como se nada tivesse acontecido

📌 **O que NÃO aconteceu:**
- ❌ Não perdemos dados (os dados estavam no HDFS, não no NodeManager)
- ❌ Não precisamos reiniciar o Job do zero
- ❌ Não houve corrupção de resultado

⚠️ **Limitação visível**: o Job provavelmente ficou **mais lento** após a falha,
porque agora só temos 1 NodeManager com 1 executor (em vez de 2). Isso é esperado
— a tolerância a falhas garante **corretude**, não desempenho.

> 💡 **Compare com a demonstração do cdn-hadoop-lab** (`restart-datanode3`):
> lá, um DataNode do HDFS morria e os blocos eram servidos pela réplica. Aqui,
> um NodeManager do YARN morre e as Tasks são recalculadas. Mecanismos diferentes
> (replicação vs linhagem), mesmo princípio de resiliência.

## Traga o nodemanager1 de volta

Agora vamos restaurar o NodeManager ao cluster:

```bash
docker start nodemanager1
```

Execute a célula abaixo para confirmar que ele retorna ao cluster. O ResourceManager
detecta automaticamente o novo nó e o adiciona ao pool de recursos disponíveis.

Verifique http://localhost:8088/cluster/nodes para ver ambos NodeManagers listados
como `RUNNING` novamente. Pode levar ~30s para o NodeManager se registrar.

In [ ]:
# Inicia o container nodemanager1 novamente
# O YARN detecta automaticamente o retorno e pode alocar novos containers nele
subprocess.run(["docker", "start", "nodemanager1"], check=True)
print("nodemanager1 restarted — give it ~30s, then check http://localhost:8088/cluster/nodes")

# Finaliza a SparkSession e libera os recursos no YARN
spark.stop()
print("✅ Chaos Lab concluído. O Spark tolerou uma falha de nó sem perder dados nem corromper resultados.")